# 🎒 Bagging Workshop: Heart Disease Prediction

In this interactive lab, we will build a Bagging ensemble from scratch, use NumPy for vectorization, and finally leverage Scikit-Learn's production-ready implementation to predict heart disease.

## 1. Problem Overview

**Goal**: Predict whether a patient has a severe clinical condition based on their features.

We will start with a baseline Decision Tree, then build Bagging to see how it reduces variance and improves accuracy.

## 2. Dataset Exploration

For this lab, we will use the Breast Cancer dataset as a proxy for clinical prediction tasks.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target)

X.head()

## 3. Data Cleaning & 4. Feature Engineering

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")

## 5. Baseline Model (Single Decision Tree)

In [ ]:
from sklearn.tree import DecisionTreeClassifier

baseline_tree = DecisionTreeClassifier(random_state=42)
baseline_tree.fit(X_train, y_train)

y_pred_base = baseline_tree.predict(X_test)
print("Baseline Decision Tree Accuracy:", accuracy_score(y_test, y_pred_base))

## 6. From Scratch Implementation

In [ ]:
from collections import Counter

class BaggingFromScratch:
    def __init__(self, n_estimators=10):
        self.n_estimators = n_estimators
        self.models = []
        
    def fit(self, X, y):
        n_samples = len(X)
        X_np, y_np = np.array(X), np.array(y)
        
        for _ in range(self.n_estimators):
            indices = np.random.choice(n_samples, size=n_samples, replace=True)
            X_boot, y_boot = X_np[indices], y_np[indices]
            
            tree = DecisionTreeClassifier(random_state=42)
            tree.fit(X_boot, y_boot)
            self.models.append(tree)
            
    def predict(self, X):
        predictions = np.array([model.predict(X) for model in self.models])
        
        final_preds = []
        for i in range(len(X)):
            votes = predictions[:, i]
            majority = Counter(votes).most_common(1)[0][0]
            final_preds.append(majority)
        return np.array(final_preds)

bagging_scratch = BaggingFromScratch(n_estimators=50)
bagging_scratch.fit(X_train, y_train)
preds_scratch = bagging_scratch.predict(X_test)
print("From Scratch Bagging Accuracy:", accuracy_score(y_test, preds_scratch))

## 7. NumPy Implementation (Vectorized)

In [ ]:
from scipy.stats import mode

class NumpyBagging:
    def __init__(self, n_estimators=50):
        self.n_estimators = n_estimators
        self.trees = []
        
    def fit(self, X, y):
        n_samples = len(X)
        X_np, y_np = np.array(X), np.array(y)
        
        boot_indices = np.random.randint(0, n_samples, size=(self.n_estimators, n_samples))
        
        for idx in boot_indices:
            tree = DecisionTreeClassifier(max_depth=None)
            tree.fit(X_np[idx], y_np[idx])
            self.trees.append(tree)
            
    def predict(self, X):
        tree_preds = np.array([tree.predict(X) for tree in self.trees])
        final_preds, _ = mode(tree_preds, axis=0, keepdims=False)
        return final_preds

np_bag = NumpyBagging(n_estimators=50)
np_bag.fit(X_train, y_train)
print("NumPy Bagging Accuracy:", accuracy_score(y_test, np_bag.predict(X_test)))

## 8. Library Implementation (Scikit-Learn)

In [ ]:
from sklearn.ensemble import BaggingClassifier

sk_bag = BaggingClassifier(estimator=DecisionTreeClassifier(), n_estimators=100, oob_score=True, random_state=42, n_jobs=-1)
sk_bag.fit(X_train, y_train)

print("OOB Score:", sk_bag.oob_score_)
print("Test Accuracy:", accuracy_score(y_test, sk_bag.predict(X_test)))

## 9. Hyperparameter Experiments

In [ ]:
estimators = [10, 50, 100, 200, 500]
scores = []

for n in estimators:
    model = BaggingClassifier(estimator=DecisionTreeClassifier(), n_estimators=n, random_state=42)
    model.fit(X_train, y_train)
    scores.append(accuracy_score(y_test, model.predict(X_test)))

plt.plot(estimators, scores, marker='o')
plt.title("Accuracy vs Number of Estimators")
plt.xlabel("n_estimators")
plt.ylabel("Accuracy")
plt.show()

## 10. Model Comparison & 11. Error Analysis

In [ ]:
print("Baseline Classification Report:\n", classification_report(y_test, y_pred_base))
print("\nBagging Classification Report:\n", classification_report(y_test, sk_bag.predict(X_test)))

## 12. Feature Importance Analysis

In [ ]:
importances = np.mean([tree.feature_importances_ for tree in sk_bag.estimators_], axis=0)
indices = np.argsort(importances)[::-1][:10]

plt.figure(figsize=(10, 5))
plt.bar(range(10), importances[indices])
plt.xticks(range(10), X.columns[indices], rotation=45)
plt.title("Top 10 Feature Importances in Bagging Ensemble")
plt.show()

## 13. Explainability (SHAP where relevant)

In [ ]:
import shap

# Explain the first tree in the ensemble
explainer = shap.TreeExplainer(sk_bag.estimators_[0])
shap_values = explainer.shap_values(X_test)
shap.summary_plot(shap_values, X_test, plot_type="bar")

## 14. Mini Challenges
1. Modify the `BaggingFromScratch` to use `LogisticRegression` as a base estimator. Does accuracy improve?
2. Implement Pasting (sampling WITHOUT replacement) in the NumPy version.

## 15. Solutions
*(Try the challenges before looking at solutions!)*

## 16. Key Takeaways
- Bagging powerfully reduces variance.
- Scikit-learn's implementation is highly optimized.
- Feature importance in bagging is derived by averaging over trees.